# Plasma Screening Simulator — LAMMPS Implementation

**Debye-Hückel Theory · Molecular Dynamics · LAMMPS**

This notebook replicates every feature of the browser-based *Plasma Screening Simulator* (`plasma_simulator.html`) using the LAMMPS MD library.

## Two parallel simulations

| Model | Particles | Potential | Electrons |
|---|---|---|---|
| **Yukawa / Debye-Hückel** | Protons only | `V(r) = A·exp(-κr)/r` | Implicit (encoded in κ = 1/λ_D) |
| **Explicit Coulomb (50/50)** | Protons + Electrons | Bare Coulomb + WCA cores | Explicit |

## Normalized unit system

All simulations run in dimensionless units normalized to the Debye scale:

| Quantity | Unit | SI value |
|---|---|---|
| Length | λ_D (Debye length) | computed below |
| Energy | k_BT | computed below |
| Mass | m_p (proton mass) | 1.673 × 10⁻²⁷ kg |
| Time | τ = √(m_p λ_D² / k_BT) | computed below |
| Charge | √(k_BT λ_D / k_e) | computed below |

## Key references
- Debye-Hückel theory: Debye & Hückel (1923)
- WCA potential: Weeks, Chandler & Andersen, *J. Chem. Phys.* 54, 5237 (1971)
- Plasma coupling: Ichimaru, *Rev. Mod. Phys.* 54, 1017 (1982)

---
## Cell 1 — Physical Constants & Plasma Parameters

Compute the Debye length λ_D, Yukawa amplitude A_norm, coupling parameter Γ, and the time unit τ from the user-specified temperature and number density.
These values are injected into the LAMMPS input scripts in the cells below.

In [ ]:
import numpy as np
from plasmapy.formulary import Debye_length
from astropy import units as u

# ── Physical Constants (SI) ───────────────────────────────────
e      = 1.602176634e-19   # [C]        elementary charge
eps0   = 8.8541878128e-12  # [F/m]      vacuum permittivity
m_p    = 1.67262192e-27    # [kg]       proton mass
m_e    = 9.10938372e-31    # [kg]       electron mass
k_B    = 1.380649e-23      # [J/K]      Boltzmann constant
k_e    = 1.0 / (4.0 * np.pi * eps0)  # [N·m²/C²]  Coulomb constant

# ── Plasma Conditions (match HTML simulator defaults) ─────────
T_eV   = 0.10              # [eV]   temperature  (HTML slider default)
n_m3   = 1e21              # [m⁻³]  number density (HTML: 10^21 m⁻³)
T_K    = T_eV * 11604.52   # [K]    temperature

# ── Debye Length ──────────────────────────────────────────────
lambda_D_m = Debye_length(T_eV * u.eV, n_m3 * u.m**-3).to(u.m).value  # [m]
lambda_D_A = lambda_D_m * 1e10                                           # [Å]

# ── Energy & Time Scales ──────────────────────────────────────
energy_scale = k_B * T_K                                    # [J]  1 k_BT
time_scale   = np.sqrt(m_p * lambda_D_m**2 / (k_B * T_K))  # [s]  1 τ

# ── Yukawa Prefactor ──────────────────────────────────────────
# V(r) = A_norm · exp(-κr) / r   [k_BT · λ_D]
# A_norm = (e²/4πε₀) / (k_B T λ_D)   (dimensionless in normalized units)
A_norm     = (e**2 / (4.0 * np.pi * eps0)) / (k_B * T_K * lambda_D_m)
kappa_norm = 1.0   # [λ_D⁻¹]  screening wavenumber (= 1 by normalization)

# ── Normalized Charge (for Explicit Coulomb sim) ──────────────
# q_eff satisfies: k_e * q_eff² / (k_B T λ_D) = A_norm
# => q_eff = sqrt(A_norm)  [normalized units]
q_eff = np.sqrt(A_norm)

# ── Simulation Parameters (matching HTML) ─────────────────────
N_p       = 50             # proton count  (HTML default)
N_e       = 50             # electron count (HTML default)
BOX       = 10.0           # [λ_D]  box side length (= 10 λ_D as in HTML)

# Yukawa simulation
CUT_YUK   = 3.0            # [λ_D]  Yukawa cutoff
DT_YUK    = 0.001          # [τ]    timestep
STEPS_YUK = 5000           # production steps → 5 τ total
DUMP_YUK  = 25             # dump every N steps (200 frames)
THERMO_YUK = 20            # velocity-rescale period [steps] (matches HTML)

# Coulomb simulation
CUT_COL   = 4.0            # [λ_D]  Coulomb cutoff
DT_COL    = 0.0001         # [τ]    timestep (10x smaller — fast electrons)
STEPS_COL = 50000          # production steps → 5 τ total
DUMP_COL  = 250            # dump every N steps (200 frames)
THERMO_COL = 20            # velocity-rescale period [steps]

# WCA core parameters (identical to HTML)
SIG_PP = 0.30              # [λ_D]  proton–proton WCA σ
SIG_EP = 0.05              # [λ_D]  electron–proton WCA σ
SIG_EE = 0.04              # [λ_D]  electron–electron WCA σ
EPS_WCA = 5.0              # [k_BT] WCA well depth
LC_PP  = 2.0**(1.0/6.0) * SIG_PP  # WCA cutoff p-p ≈ 0.337 λ_D
LC_EP  = 2.0**(1.0/6.0) * SIG_EP  # WCA cutoff e-p ≈ 0.0561 λ_D
LC_EE  = 2.0**(1.0/6.0) * SIG_EE  # WCA cutoff e-e ≈ 0.0449 λ_D

# Mass ratio (HTML uses 1:100 for numerical stability, not 1:1836)
ME_R      = 100.0          # proton-to-electron mass ratio in sim
MASS_E    = 1.0 / ME_R     # [m_p]  electron mass in simulation

# Coupling parameter (2D Wigner-Seitz radius)
r_WS_2D   = np.sqrt(BOX**2 / (np.pi * N_p))  # 2D Wigner-Seitz radius [λ_D]
Gamma     = A_norm / r_WS_2D                   # coupling parameter Γ

# ── Print Summary ─────────────────────────────────────────────
print("=" * 65)
print("  PLASMA SCREENING — UNIT REFERENCE")
print("=" * 65)
print(f"\n[Normalization Basis]")
print(f"  Length  (1 λ_D)  : {lambda_D_m:.4e} m  =  {lambda_D_A:.4f} Å")
print(f"  Energy  (1 k_BT) : {energy_scale:.4e} J  =  {energy_scale/e:.6f} eV")
print(f"  Mass    (1 m_p)  : {m_p:.4e} kg  =  1.007 amu")
print(f"  Time    (1 τ)    : {time_scale:.4e} s  =  {time_scale*1e12:.4f} ps")
print(f"\n[Plasma Conditions]")
print(f"  Temperature      : {T_eV} eV  =  {T_K:.2f} K")
print(f"  Number Density   : {n_m3:.2e} m⁻³")
print(f"  Debye Length λ_D : {lambda_D_m:.4e} m  =  {lambda_D_A:.4f} Å")
print(f"\n[Simulation Parameters]")
print(f"  Protons N_p      : {N_p}")
print(f"  Electrons N_e    : {N_e}")
print(f"  Box Side         : {BOX} λ_D  =  {BOX*lambda_D_A:.2f} Å")
print(f"  Yukawa κ         : {kappa_norm} λ_D⁻¹  (= 1/λ_D by definition)")
print(f"  Yukawa A (norm)  : {A_norm:.6f}  [e²/4πε₀k_BTλ_D]")
print(f"  Normalized charge: {q_eff:.6f}  [= √A_norm]")
print(f"  Coupling Γ       : {Gamma:.5f}  ({'weakly coupled Γ<<1' if Gamma < 1 else 'strongly coupled Γ>>1'})")
print(f"  Yukawa dt        : {DT_YUK} τ  =  {DT_YUK*time_scale*1e15:.4f} fs")
print(f"  Coulomb dt       : {DT_COL} τ  =  {DT_COL*time_scale*1e15:.4f} fs")
print(f"  Electron mass    : 1/{ME_R:.0f} m_p  (HTML uses 1:100, not 1:1836)")

---
## Cell 2 — Yukawa / Debye-Hückel LAMMPS Simulation

**Model:** Protons only. Electrons are implicit — their screening is entirely encoded in the Yukawa screening parameter κ = 1/λ_D.

**Potential:** `V(r) = A · exp(-κr) / r`  [k_BT]

This corresponds to LAMMPS `pair_style yukawa` with κ = 1 (in normalized units).

**Thermostat:** Velocity-rescaling every 20 steps (matches HTML simulator).

**Integrator:** Velocity-Verlet (NVE) — LAMMPS default.

**Geometry:** 2D periodic box of side 10 λ_D.

In [ ]:
from lammps import lammps
import os

lmp_yuk = lammps()

lmp_yuk.commands_string(f"""
# ══════════════════════════════════════════════════════════════════
#  PLASMA SCREENING — YUKAWA (DEBYE-HÜCKEL) SIMULATION
#
#  Unit system : normalized  (length = λ_D, energy = k_BT, mass = m_p)
#  Dimensionality : 2-D periodic (matches HTML browser simulation)
#
#  Normalization basis:
#    Length  → λ_D  = {lambda_D_m:.4e} m  =  {lambda_D_A:.4f} Å
#    Energy  → k_BT = {energy_scale:.4e} J  =  {energy_scale/e:.6f} eV
#    Mass    → m_p  = {m_p:.4e} kg
#    Time    → τ    = {time_scale:.4e} s  =  {time_scale*1e12:.4f} ps
#
#  Yukawa potential: V(r) = A·exp(-κr)/r  [k_BT]
#    A  = {A_norm:.6f}  [e²/4πε₀k_BTλ_D,  dimensionless]
#    κ  = {kappa_norm}    [λ_D⁻¹,  = 1/λ_D by normalization]
#  Electrons implicit — encoded in κ (Debye-Hückel theory).
# ══════════════════════════════════════════════════════════════════

# ── Dimensionality & Units ─────────────────────────────────────
dimension     2
units         lj                          # normalized  (see above for SI)
atom_style    atomic                      # id type x[λ_D] y[λ_D]
boundary      p p p                       # periodic in x, y (z irrelevant in 2D)

# ── Simulation Box & Atoms ────────────────────────────────────
# Box: {BOX} λ_D × {BOX} λ_D  =  {BOX*lambda_D_A:.2f} Å × {BOX*lambda_D_A:.2f} Å
region        box block 0 {BOX} 0 {BOX} -0.5 0.5
create_box    1 box                       # 1 atom type: protons
create_atoms  1 random {N_p} 12345 box overlap 0.25 maxtry 5000

# ── Mass  [m_p] ───────────────────────────────────────────────
mass          1 1.0                       # proton mass  =  1.0 m_p

# ── Neighbor List ─────────────────────────────────────────────
neigh_modify  one 5000 delay 0 every 1 check yes

# ── Stage 1: WCA Soft-Wall Pre-equilibration ──────────────────
# Push any overlapping protons apart before turning on Yukawa.
pair_style    soft 1.0
pair_coeff    * * 10.0
velocity      all create 1.0 54321 dist gaussian
fix           warmup all nve/limit 0.05
run           500
unfix         warmup

# ── Stage 2: Yukawa Production Run ────────────────────────────
# V(r) = A·exp(-κ·r)/r  [k_BT]
# κ = {kappa_norm} [λ_D⁻¹]   cutoff = {CUT_YUK} [λ_D]   (e^-{CUT_YUK:.0f} ≈ 5% at cutoff)
pair_style    yukawa {kappa_norm} {CUT_YUK}   # kappa[λ_D⁻¹]  cutoff[λ_D]
pair_coeff    1 1 {A_norm:.6f}               # A[dimensionless] = e²/4πε₀k_BTλ_D

# Velocity-rescaling thermostat — rescales every {THERMO_YUK} steps to T=1 k_BT
# This matches the HTML simulator's rescale-every-20-steps approach.
fix           1 all nve
fix           2 all temp/rescale {THERMO_YUK} 1.0 1.0 0.05 1.0

# ── Output ─────────────────────────────────────────────────────
reset_timestep 0
thermo_style  custom step temp ke pe etotal press
thermo        {DUMP_YUK}

# Dump: id type x y z vx vy  (positions in [λ_D], velocities in [λ_D/τ])
dump          1 all custom {DUMP_YUK} yukawa.dump id type x y z vx vy
dump_modify   1 sort id

# ── Run ────────────────────────────────────────────────────────
timestep      {DT_YUK}                    # [τ] = {DT_YUK*time_scale*1e15:.4f} fs
run           {STEPS_YUK}                 # {STEPS_YUK} steps = {STEPS_YUK*DT_YUK:.1f} τ
""")

lmp_yuk.close()
print(f"✓ Yukawa LAMMPS simulation complete")
print(f"  Timestep        : {DT_YUK} τ  =  {DT_YUK*time_scale*1e15:.4f} fs")
print(f"  Total time      : {STEPS_YUK*DT_YUK:.2f} τ  =  {STEPS_YUK*DT_YUK*time_scale*1e12:.4f} ps")
print(f"  Frames written  : {STEPS_YUK // DUMP_YUK}  →  yukawa.dump")

---
## Cell 3 — Explicit Coulomb (50/50) LAMMPS Simulation

**Model:** Equal numbers of protons and electrons. Both species interact via bare Coulomb plus WCA short-range repulsive cores.

**Potential:**
- **Coulomb (all pairs):** `V_C(r) = q_i q_j / r`  (LAMMPS `coul/cut`, cutoff = 4 λ_D)
- **WCA core (all pairs, r < r_cut):** `V_WCA(r) = 4ε[(σ/r)¹² − (σ/r)⁶]`, cutoff at r = 2^(1/6)σ  
  - p–p: σ = 0.30 λ_D  (prevents hard proton overlap)  
  - e–p: σ = 0.05 λ_D  (mimics quantum zero-point pressure — prevents classical collapse)  
  - e–e: σ = 0.04 λ_D  (electron–electron repulsion hard core)

**Mass ratio:** 1:100 (HTML uses 1:100 for numerical stability; 1:1836 would require dt ~ 10⁻⁶ τ).

**Thermostat:** Velocity-rescaling every 20 steps, applied separately to protons and electrons (matches HTML).

In [ ]:
from lammps import lammps

lmp_col = lammps()

lmp_col.commands_string(f"""
# ══════════════════════════════════════════════════════════════════
#  PLASMA SCREENING — EXPLICIT COULOMB (50/50) SIMULATION
#
#  Unit system : normalized  (length = λ_D, energy = k_BT, mass = m_p)
#  Dimensionality : 2-D periodic (matches HTML browser simulation)
#
#  Particles:
#    type 1 = protons    (mass = 1.0 m_p,      charge = +{q_eff:.6f})
#    type 2 = electrons  (mass = {MASS_E:.6f} m_p,  charge = -{q_eff:.6f})
#
#  Potential: V = V_Coulomb + V_WCA
#    V_C(r)   = q_i·q_j / r     (cutoff {CUT_COL} λ_D)
#    V_WCA(r) = 4ε[(σ/r)¹²−(σ/r)⁶]  (r < 2^(1/6)·σ, ε={EPS_WCA})
#      σ_pp = {SIG_PP}  σ_ep = {SIG_EP}  σ_ee = {SIG_EE}   [λ_D]
# ══════════════════════════════════════════════════════════════════

# ── Dimensionality & Units ─────────────────────────────────────
dimension     2
units         lj
atom_style    charge                      # id type x y z vx vy charge
boundary      p p p

# ── Simulation Box ────────────────────────────────────────────
region        box block 0 {BOX} 0 {BOX} -0.5 0.5
create_box    2 box

# Create protons (type 1) and electrons (type 2) on separate grids
# overlap=0.5 ensures minimum separation ≥ 0.5 λ_D between all atoms
create_atoms  1 random {N_p} 12345 box overlap 0.5 maxtry 5000
create_atoms  2 random {N_e} 67890 box overlap 0.5 maxtry 5000

# ── Masses [m_p] ──────────────────────────────────────────────
mass          1 1.0                       # proton  mass = m_p
mass          2 {MASS_E:.6f}              # electron mass = 1/{ME_R:.0f} m_p

# ── Charges [normalized] ──────────────────────────────────────
set           type 1 charge +{q_eff:.6f}  # proton charge
set           type 2 charge -{q_eff:.6f}  # electron charge

# ── Neighbor List ─────────────────────────────────────────────
neigh_modify  one 5000 delay 0 every 1 check yes

# ── Stage 1: Soft-Wall Pre-equilibration ──────────────────────
# Gently separate overlapping atoms before activating Coulomb.
pair_style    soft 1.0
pair_coeff    * * 5.0
velocity      all create 1.0 42 dist gaussian
fix           warmup all nve/limit 0.02
run           1000
unfix         warmup

# ── Stage 2: Coulomb + WCA Production Run ─────────────────────
#
# pair_style hybrid/overlay:
#   lj/cut   → WCA repulsive cores (truncated at 2^(1/6)·σ per pair)
#   coul/cut → bare Coulomb (cutoff = {CUT_COL} λ_D)
#
# Global lj/cut cutoff = max(LC_PP, LC_EP, LC_EE) + small buffer
pair_style    hybrid/overlay lj/cut {LC_PP*1.05:.6f} coul/cut {CUT_COL}

# WCA p-p: prevents proton hard overlap
pair_coeff    1 1 lj/cut {EPS_WCA} {SIG_PP} {LC_PP:.6f}
# WCA e-p: prevents classical electron-proton collapse
#          (mimics quantum zero-point pressure)
pair_coeff    1 2 lj/cut {EPS_WCA} {SIG_EP} {LC_EP:.6f}
# WCA e-e: prevents electron overlap, makes e-e repulsion visible
pair_coeff    2 2 lj/cut {EPS_WCA} {SIG_EE} {LC_EE:.6f}

# Coulomb between all pairs (uses global cutoff = {CUT_COL} λ_D)
pair_coeff    1 1 coul/cut
pair_coeff    1 2 coul/cut
pair_coeff    2 2 coul/cut

# Energy minimization to relax any residual overlaps before dynamics
min_style     cg
minimize      1.0e-4 1.0e-6 500 5000

# ── Integrator + Thermostat ───────────────────────────────────
reset_timestep 0
velocity      all create 1.0 99 dist gaussian

# NVE integrator (velocity-Verlet)
fix           1 all nve

# Velocity-rescaling thermostat: rescale protons and electrons independently
# every {THERMO_COL} steps to T = 1.0 (= k_BT).  Matches HTML's _thermo() method.
fix           2 all temp/rescale {THERMO_COL} 1.0 1.0 0.05 1.0

# ── Output ────────────────────────────────────────────────────
thermo_style  custom step temp ke pe etotal press
thermo        {DUMP_COL}

# Dump: id type x y z vx vy  (positions in [λ_D])
dump          1 all custom {DUMP_COL} coulomb.dump id type x y z vx vy
dump_modify   1 sort id

# ── Run ───────────────────────────────────────────────────────
timestep      {DT_COL}                    # [τ] = {DT_COL*time_scale*1e15:.4f} fs
run           {STEPS_COL}                 # {STEPS_COL} steps = {STEPS_COL*DT_COL:.1f} τ
""")

lmp_col.close()
print(f"✓ Explicit Coulomb LAMMPS simulation complete")
print(f"  Timestep        : {DT_COL} τ  =  {DT_COL*time_scale*1e15:.4f} fs")
print(f"  Total time      : {STEPS_COL*DT_COL:.1f} τ  =  {STEPS_COL*DT_COL*time_scale*1e12:.4f} ps")
print(f"  Frames written  : {STEPS_COL // DUMP_COL}  →  coulomb.dump")

---
## Cell 4 — Parse LAMMPS Dump Files

Read the positions (and optionally velocities) from `yukawa.dump` and `coulomb.dump`.

In [ ]:
import numpy as np

def parse_lammps_dump(filepath):
    """
    Parse a LAMMPS custom dump file.

    Returns
    -------
    frames_pos   : list of (N, 3) arrays  — positions [λ_D] per frame
    frames_types : list of (N,)   arrays  — atom type per frame
    """
    frames_pos   = []
    frames_types = []

    with open(filepath, "r") as fh:
        lines = fh.readlines()

    i = 0
    while i < len(lines):
        if "ITEM: TIMESTEP" in lines[i]:
            i += 2                     # skip timestep value
            i += 2                     # skip NUMBER OF ATOMS header + count
            i += 4                     # skip BOX BOUNDS header + 3 lines
            i += 1                     # skip ITEM: ATOMS header

            pos   = []
            types = []
            while i < len(lines) and "ITEM:" not in lines[i]:
                parts = lines[i].split()
                # columns: id type x y z vx vy
                types.append(int(parts[1]))
                pos.append([float(parts[2]), float(parts[3]), float(parts[4])])
                i += 1

            frames_pos.append(np.array(pos))
            frames_types.append(np.array(types))
        else:
            i += 1

    return frames_pos, frames_types


# ── Parse Yukawa dump ─────────────────────────────────────────
yuk_pos, yuk_types = parse_lammps_dump("yukawa.dump")
# Skip frame 0 (pre-run initial state)
yuk_pos   = yuk_pos[1:]
yuk_types = yuk_types[1:]

# ── Parse Coulomb dump ────────────────────────────────────────
col_pos, col_types = parse_lammps_dump("coulomb.dump")
col_pos   = col_pos[1:]
col_types = col_types[1:]

print(f"✓ Yukawa frames    : {len(yuk_pos)}  ({len(yuk_pos[0])} protons/frame)")
print(f"✓ Coulomb frames   : {len(col_pos)}  ({np.sum(col_types[0]==1)} protons + {np.sum(col_types[0]==2)} electrons/frame)")
print(f"  Box side         : {BOX} λ_D")

---
## Cell 5 — Radial Distribution Functions (RDF)

Compute g(r) for:
- **Yukawa:** proton–proton  (g < 1 at small r shows Yukawa repulsion)
- **Coulomb:** electron–proton  (g > 1 at small r shows Debye cloud forming)

The normalization follows the **2D RDF** formula (identical to the HTML simulator):

$$g(r) = \frac{\text{hist}[r]}{N^2 \cdot \frac{1}{L^2} \cdot 2\pi r \,\Delta r \cdot N_{\text{frames}}}$$

In [ ]:
import numpy as np

def min_image_2d(delta, box):
    """Apply minimum-image convention for periodic boundaries."""
    delta = delta - box * np.round(delta / box)
    return delta


def compute_rdf_pp(frames_pos, box, n_bins=60, r_max=None):
    """
    Proton-proton 2D RDF g(r), accumulating over all frames.

    Parameters
    ----------
    frames_pos : list of (N, 3) arrays  — positions [λ_D]
    box        : float  — box side length [λ_D]
    n_bins     : int    — number of histogram bins (HTML default: 60)
    r_max      : float  — maximum r [λ_D]  (default: CUT_YUK * 0.9)

    Returns
    -------
    r_centers : (n_bins,) array  — bin center positions [λ_D]
    g_r       : (n_bins,) array  — g(r) values
    """
    if r_max is None:
        r_max = CUT_YUK * 0.9   # matches HTML: rmax = cutoff * 0.9
    dr      = r_max / n_bins
    hist    = np.zeros(n_bins)
    n_frames = len(frames_pos)

    for pos in frames_pos:
        N  = len(pos)
        xy = pos[:, :2]          # use x, y only (2D)
        for i in range(N - 1):
            dx = min_image_2d(xy[i+1:, 0] - xy[i, 0], box)
            dy = min_image_2d(xy[i+1:, 1] - xy[i, 1], box)
            r  = np.sqrt(dx**2 + dy**2)
            # Exclude r < 0.02 (very close — numerical artifact)
            mask = (r > 0.02) & (r < r_max)
            b    = (r[mask] / dr).astype(int)
            b    = np.clip(b, 0, n_bins - 1)
            np.add.at(hist, b, 1)

    # Normalize using 2D RDF formula (matches HTML accumulator)
    N       = len(frames_pos[0])
    dens    = N / (box * box)   # 2D number density
    r_cents = np.arange(n_bins) * dr + 0.5 * dr
    area    = 2.0 * np.pi * r_cents * dr
    exp_cnt = dens * area * N * n_frames   # expected counts (uniform gas)
    g_r     = np.where(exp_cnt > 0, hist / exp_cnt, 0.0)

    return r_cents, g_r


def compute_rdf_ep(frames_pos, frames_types, box, n_bins=60, r_max=None):
    """
    Electron-proton 2D RDF g(r), accumulating over all frames.

    Parameters
    ----------
    frames_pos   : list of (N, 3) arrays  — positions [λ_D]
    frames_types : list of (N,)   arrays  — atom types (1=proton, 2=electron)
    box          : float  — box side length [λ_D]
    n_bins       : int    — number of histogram bins
    r_max        : float  — maximum r [λ_D]  (default: CUT_COL * 0.9)

    Returns
    -------
    r_centers : (n_bins,) array  — bin center positions [λ_D]
    g_r       : (n_bins,) array  — g(r) values
    """
    if r_max is None:
        r_max = CUT_COL * 0.9   # matches HTML: rmax = cutoff * 0.9
    dr      = r_max / n_bins
    hist    = np.zeros(n_bins)
    n_frames = len(frames_pos)

    for pos, types in zip(frames_pos, frames_types):
        p_mask = (types == 1)
        e_mask = (types == 2)
        p_xy   = pos[p_mask, :2]   # proton positions
        e_xy   = pos[e_mask, :2]   # electron positions

        # Iterate electrons as reference, protons as targets
        # (matches HTML: i=electron, j=proton)
        for e in e_xy:
            dx = min_image_2d(p_xy[:, 0] - e[0], box)
            dy = min_image_2d(p_xy[:, 1] - e[1], box)
            r  = np.sqrt(dx**2 + dy**2)
            mask = (r > 0.02) & (r < r_max)
            b    = (r[mask] / dr).astype(int)
            b    = np.clip(b, 0, n_bins - 1)
            np.add.at(hist, b, 1)

    # Normalize using HTML's formula: exp = (N_p/box²) · 2πr·dr · N_p · cnt
    N_p_sim = int(np.sum(frames_types[0] == 1))
    dens    = N_p_sim / (box * box)
    r_cents = np.arange(n_bins) * dr + 0.5 * dr
    area    = 2.0 * np.pi * r_cents * dr
    exp_cnt = dens * area * N_p_sim * n_frames
    g_r     = np.where(exp_cnt > 0, hist / exp_cnt, 0.0)

    return r_cents, g_r


# ── Compute RDFs ──────────────────────────────────────────────
yuk_rc, yuk_gr = compute_rdf_pp(yuk_pos, BOX)
col_rc, col_gr = compute_rdf_ep(col_pos, col_types, BOX)

# ── Peak statistics (match HTML display) ─────────────────────
yuk_peak_idx = np.argmax(yuk_gr)
col_peak_idx = np.argmax(col_gr)

print("=" * 55)
print("  RDF STATISTICS")
print("=" * 55)
print(f"\n[Yukawa — Proton-Proton g(r)]")
print(f"  Peak g(r) : {yuk_gr[yuk_peak_idx]:.4f}")
print(f"  Peak at r : {yuk_rc[yuk_peak_idx]:.4f} λ_D")
print(f"  Frames    : {len(yuk_pos)}")
print(f"\n[Coulomb — Electron-Proton g(r)]")
print(f"  Peak g(r) : {col_gr[col_peak_idx]:.4f}")
print(f"  Peak at r : {col_rc[col_peak_idx]:.4f} λ_D")
print(f"  Frames    : {len(col_pos)}")
print()
print(f"  → Yukawa  g<1 at small r : repulsion expected (g < 1 means deficit)")
print(f"  → Coulomb g>1 at small r : Debye cloud expected (g > 1 means excess)")

---
## Cell 6 — Visualization

Reproduce the full HTML simulator display in matplotlib:

1. **Top row:** 2D particle snapshots (latest frame)
   - Yukawa: protons (red dots) with Debye-cloud halo
   - Coulomb: protons (red, large) + electrons (purple, small)

2. **Bottom row:** RDF plots
   - g=1 reference line (dashed)
   - Shaded region above/below g=1
   - Peak annotation

3. **Stats bar:** λ_D, Γ, A_norm, box size, timesteps, frame count

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

# ── Style ─────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor" : "#030b14",
    "axes.facecolor"   : "#020910",
    "axes.edgecolor"   : "#1a3a5c",
    "axes.labelcolor"  : "#4a7090",
    "xtick.color"      : "#4a7090",
    "ytick.color"      : "#4a7090",
    "text.color"       : "#cce0f0",
    "grid.color"       : "#1a3a5c",
    "grid.linestyle"   : "--",
    "grid.alpha"       : 0.3,
    "font.family"      : "monospace",
    "font.size"        : 9,
})

BLUE   = "#00c8f0"
PURPLE = "#9f7aea"
RED    = "#f05060"
GOLD   = "#f0c040"

# ── Use last frame for particle snapshots ─────────────────────
yuk_snap  = yuk_pos[-1][:, :2]   # (N_p, 2) proton positions
col_snap  = col_pos[-1][:, :2]   # (N, 2) all positions
col_t     = col_types[-1]        # (N,)  types
p_snap    = col_snap[col_t == 1] # proton positions
e_snap    = col_snap[col_t == 2] # electron positions


def plot_rdf(ax, r_centers, g_r, color, label, mode, peak_r=None, peak_g=None):
    """Render an RDF panel matching the HTML simulator's renderRDF() function."""
    # g=1 reference line
    ax.axhline(1.0, color="white", lw=1.0, ls="--", alpha=0.4, label="g = 1")
    ax.text(r_centers[-1] * 0.98, 1.03, "g=1", color="white", alpha=0.5,
            ha="right", va="bottom", fontsize=8)

    # Shaded fill: excess (g>1) and deficit (g<1) regions
    ax.fill_between(r_centers, 1.0, g_r,
                    where=(g_r >= 1.0),
                    color=color, alpha=0.18, label="excess (g>1)")
    ax.fill_between(r_centers, 1.0, g_r,
                    where=(g_r < 1.0),
                    color=RED, alpha=0.18, label="deficit (g<1)")

    # Main g(r) curve
    ax.plot(r_centers, g_r, color=color, lw=2.0)

    # Peak annotation
    if peak_r is not None and peak_g is not None:
        ax.annotate(
            f"peak {peak_g:.3f}\nr = {peak_r:.3f} λ_D",
            xy=(peak_r, peak_g),
            xytext=(peak_r + r_centers[-1] * 0.12, peak_g * 0.88),
            color=color, fontsize=8,
            arrowprops=dict(arrowstyle="->", color=color, lw=0.8),
        )

    ax.set_xlim(0, r_centers[-1])
    ax.set_ylim(0, max(2.5, np.nanmax(g_r[np.isfinite(g_r)]) * 1.1))
    ax.set_xlabel("r  (λ_D)", fontsize=9)
    ax.set_ylabel("g(r)",     fontsize=9)
    ax.set_title(label, color=color, fontsize=9, pad=5)
    ax.text(0.02, 0.97, mode, transform=ax.transAxes,
            color=color, fontsize=8, va="top", alpha=0.85)
    ax.grid(True, alpha=0.25)


# ── Figure Layout ─────────────────────────────────────────────
fig = plt.figure(figsize=(14, 11), facecolor="#030b14")
fig.suptitle(
    "PLASMA SCREENING SIMULATOR — LAMMPS",
    fontsize=14, fontweight="bold", color=BLUE,
    fontfamily="monospace", y=0.98,
)
fig.text(
    0.5, 0.96,
    "Debye-Hückel Theory  ·  Molecular Dynamics  ·  LAMMPS",
    ha="center", color="#4a7090", fontsize=8,
)

gs = gridspec.GridSpec(
    3, 2,
    figure=fig,
    height_ratios=[3, 2, 0.35],
    hspace=0.40, wspace=0.22,
    left=0.06, right=0.97, top=0.94, bottom=0.04,
)

# ── Panel A: Yukawa particle positions ────────────────────────
ax_yp = fig.add_subplot(gs[0, 0])
ax_yp.set_facecolor("#020910")
ax_yp.set_aspect("equal")

# Debye-cloud halo (scatter with large, translucent marker)
ax_yp.scatter(
    yuk_snap[:, 0], yuk_snap[:, 1],
    s=350, c=BLUE, alpha=0.05, linewidths=0,
)
# Protons
ax_yp.scatter(
    yuk_snap[:, 0], yuk_snap[:, 1],
    s=18, c=RED, zorder=5, linewidths=0,
    label=f"{N_p} protons",
)
ax_yp.set_xlim(0, BOX); ax_yp.set_ylim(0, BOX)
ax_yp.set_xlabel("x  (λ_D)"); ax_yp.set_ylabel("y  (λ_D)")
ax_yp.set_title(
    f"Yukawa / Debye-Hückel  ·  {N_p} protons  ·  electrons implicit",
    color=BLUE, fontsize=9,
)
ax_yp.text(
    0.01, 0.98,
    f"Frame {len(yuk_pos)}  |  step {len(yuk_pos)*DUMP_YUK}",
    transform=ax_yp.transAxes, color=BLUE, fontsize=7.5, va="top", alpha=0.7,
)

# ── Panel B: Explicit Coulomb particle positions ───────────────
ax_cp = fig.add_subplot(gs[0, 1])
ax_cp.set_facecolor("#020910")
ax_cp.set_aspect("equal")

# Electrons
ax_cp.scatter(
    e_snap[:, 0], e_snap[:, 1],
    s=8, c=PURPLE, alpha=0.85, linewidths=0, zorder=3,
    label=f"{N_e} electrons",
)
# Protons
ax_cp.scatter(
    p_snap[:, 0], p_snap[:, 1],
    s=28, c=RED, zorder=5, linewidths=0,
    label=f"{N_p} protons",
)
ax_cp.set_xlim(0, BOX); ax_cp.set_ylim(0, BOX)
ax_cp.set_xlabel("x  (λ_D)"); ax_cp.set_ylabel("y  (λ_D)")
ax_cp.set_title(
    f"Explicit Coulomb (50/50)  ·  {N_p}p + {N_e}e  ·  WCA cores",
    color=PURPLE, fontsize=9,
)
ax_cp.legend(
    loc="upper right", fontsize=7,
    facecolor="#071525", edgecolor="#1a3a5c",
    labelcolor="white",
)
ax_cp.text(
    0.01, 0.98,
    f"Frame {len(col_pos)}  |  step {len(col_pos)*DUMP_COL}",
    transform=ax_cp.transAxes, color=PURPLE, fontsize=7.5, va="top", alpha=0.7,
)

# ── Panel C: Yukawa RDF ───────────────────────────────────────
ax_yr = fig.add_subplot(gs[1, 0])
plot_rdf(
    ax_yr, yuk_rc, yuk_gr,
    color=BLUE,
    label="RDF — Yukawa / Debye-Hückel",
    mode="● proton–proton",
    peak_r=yuk_rc[yuk_peak_idx],
    peak_g=yuk_gr[yuk_peak_idx],
)
ax_yr.text(
    0.98, 0.92,
    "g<1 at small r → Yukawa repulsion",
    transform=ax_yr.transAxes, ha="right", color="#4a7090", fontsize=7.5,
)

# ── Panel D: Coulomb RDF ──────────────────────────────────────
ax_cr = fig.add_subplot(gs[1, 1])
plot_rdf(
    ax_cr, col_rc, col_gr,
    color=PURPLE,
    label="RDF — Explicit Coulomb (50/50)",
    mode="● electron–proton",
    peak_r=col_rc[col_peak_idx],
    peak_g=col_gr[col_peak_idx],
)
ax_cr.text(
    0.98, 0.92,
    "g>1 at small r → Debye cloud forming",
    transform=ax_cr.transAxes, ha="right", color="#4a7090", fontsize=7.5,
)

# ── Stats Bar (bottom row — spans both columns) ───────────────
ax_st = fig.add_subplot(gs[2, :])
ax_st.axis("off")

stats = [
    ("Debye Length λ_D",  f"{lambda_D_m:.3e} m"),
    ("Coupling Γ",         f"{Gamma:.4f}"),
    ("Yukawa A (norm)",    f"{A_norm:.4f}"),
    ("Box Size",           f"{BOX:.1f} λ_D"),
    ("Yukawa dt",          f"{DT_YUK} τ"),
    ("Coulomb dt",         f"{DT_COL} τ"),
    ("Yukawa frames",      str(len(yuk_pos))),
    ("Coulomb frames",     str(len(col_pos))),
]

n_st = len(stats)
for k, (lbl, val) in enumerate(stats):
    x = k / n_st + 0.5 / n_st
    ax_st.text(x, 0.72, lbl.upper(), ha="center", va="center",
               color="#4a7090", fontsize=6.5,
               transform=ax_st.transAxes)
    ax_st.text(x, 0.20, val, ha="center", va="center",
               color=BLUE, fontsize=8, fontweight="bold",
               transform=ax_st.transAxes)

plt.savefig("plasma_screening_lammps.png", dpi=150, bbox_inches="tight",
            facecolor="#030b14")
plt.show()
print("✓ Figure saved → plasma_screening_lammps.png")

---
## Cell 7 — Animated Visualization (optional)

Produce a side-by-side animation of both simulations, combining 2D particle positions with RDF panels — matching the HTML simulator's video download feature.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.gridspec as gridspec
import numpy as np
from IPython.display import HTML

# ── Parameters ────────────────────────────────────────────────
# Downsample to at most 100 frames for a smooth, manageable animation
MAX_ANIM_FRAMES = min(100, len(yuk_pos), len(col_pos))
yuk_idx = np.linspace(0, len(yuk_pos) - 1, MAX_ANIM_FRAMES, dtype=int)
col_idx = np.linspace(0, len(col_pos) - 1, MAX_ANIM_FRAMES, dtype=int)

# Pre-compute RDFs for every frame to animate them
# (reuses the final accumulated RDFs for the plot — consistent with HTML)

fig = plt.figure(figsize=(13, 9), facecolor="#030b14")
gs  = gridspec.GridSpec(
    2, 2, figure=fig,
    height_ratios=[2, 1.2],
    hspace=0.42, wspace=0.22,
    left=0.06, right=0.97, top=0.93, bottom=0.07,
)

ax_yp2 = fig.add_subplot(gs[0, 0])
ax_cp2 = fig.add_subplot(gs[0, 1])
ax_yr2 = fig.add_subplot(gs[1, 0])
ax_cr2 = fig.add_subplot(gs[1, 1])

for ax in [ax_yp2, ax_cp2, ax_yr2, ax_cr2]:
    ax.set_facecolor("#020910")

fig.suptitle(
    "PLASMA SCREENING SIMULATOR — LAMMPS",
    fontsize=13, color=BLUE, fontfamily="monospace",
)

# ── Initial artists ───────────────────────────────────────────
sc_yuk_halo = ax_yp2.scatter([], [], s=300, c=BLUE, alpha=0.06, linewidths=0)
sc_yuk      = ax_yp2.scatter([], [], s=14, c=RED, zorder=5, linewidths=0)
ax_yp2.set_xlim(0, BOX); ax_yp2.set_ylim(0, BOX)
ax_yp2.set_title(f"Yukawa / Debye-Hückel  ·  {N_p} protons",
                 color=BLUE, fontsize=9)
ax_yp2.set_xlabel("x (λ_D)"); ax_yp2.set_ylabel("y (λ_D)")
txt_yuk = ax_yp2.text(0.01, 0.97, "", transform=ax_yp2.transAxes,
                      color=BLUE, fontsize=7, va="top")

sc_ele = ax_cp2.scatter([], [], s=6, c=PURPLE, alpha=0.85, linewidths=0)
sc_pro = ax_cp2.scatter([], [], s=22, c=RED, zorder=5, linewidths=0)
ax_cp2.set_xlim(0, BOX); ax_cp2.set_ylim(0, BOX)
ax_cp2.set_title(f"Explicit Coulomb  ·  {N_p}p + {N_e}e",
                 color=PURPLE, fontsize=9)
ax_cp2.set_xlabel("x (λ_D)"); ax_cp2.set_ylabel("y (λ_D)")
txt_col = ax_cp2.text(0.01, 0.97, "", transform=ax_cp2.transAxes,
                      color=PURPLE, fontsize=7, va="top")

# RDF panels — show the accumulated g(r) (same as HTML after equilibration)
ax_yr2.axhline(1.0, color="white", lw=1.0, ls="--", alpha=0.4)
line_yr, = ax_yr2.plot(yuk_rc, yuk_gr, color=BLUE, lw=2)
fill_yr_exc = ax_yr2.fill_between(yuk_rc, 1.0, yuk_gr,
                                   where=(yuk_gr >= 1.0),
                                   color=BLUE, alpha=0.18)
fill_yr_def = ax_yr2.fill_between(yuk_rc, 1.0, yuk_gr,
                                   where=(yuk_gr < 1.0),
                                   color=RED, alpha=0.18)
ax_yr2.set_xlim(0, yuk_rc[-1])
ax_yr2.set_ylim(0, max(2.5, np.nanmax(yuk_gr) * 1.1))
ax_yr2.set_xlabel("r  (λ_D)"); ax_yr2.set_ylabel("g(r)")
ax_yr2.set_title("RDF — proton–proton", color=BLUE, fontsize=9)
ax_yr2.axhline(1.0, color="white", lw=1, ls="--", alpha=0.3)
ax_yr2.grid(True, alpha=0.2)

ax_cr2.axhline(1.0, color="white", lw=1.0, ls="--", alpha=0.4)
line_cr, = ax_cr2.plot(col_rc, col_gr, color=PURPLE, lw=2)
ax_cr2.fill_between(col_rc, 1.0, col_gr,
                    where=(col_gr >= 1.0), color=PURPLE, alpha=0.18)
ax_cr2.fill_between(col_rc, 1.0, col_gr,
                    where=(col_gr < 1.0), color=RED, alpha=0.18)
ax_cr2.set_xlim(0, col_rc[-1])
ax_cr2.set_ylim(0, max(2.5, np.nanmax(col_gr) * 1.1))
ax_cr2.set_xlabel("r  (λ_D)"); ax_cr2.set_ylabel("g(r)")
ax_cr2.set_title("RDF — electron–proton", color=PURPLE, fontsize=9)
ax_cr2.grid(True, alpha=0.2)


def update(frame_idx):
    fi_y = yuk_idx[frame_idx]
    fi_c = col_idx[frame_idx]

    yp = yuk_pos[fi_y][:, :2]
    sc_yuk_halo.set_offsets(yp)
    sc_yuk.set_offsets(yp)
    txt_yuk.set_text(f"frame {fi_y+1} / step {(fi_y+1)*DUMP_YUK}")

    cp  = col_pos[fi_c][:, :2]
    ct  = col_types[fi_c]
    sc_pro.set_offsets(cp[ct == 1])
    sc_ele.set_offsets(cp[ct == 2])
    txt_col.set_text(f"frame {fi_c+1} / step {(fi_c+1)*DUMP_COL}")

    return sc_yuk_halo, sc_yuk, sc_pro, sc_ele, txt_yuk, txt_col


ani = animation.FuncAnimation(
    fig, update,
    frames=MAX_ANIM_FRAMES,
    interval=50,   # ms per frame → 20 fps
    blit=True,
)

# Save as MP4 (requires ffmpeg) — matches HTML's video download feature
try:
    ani.save("plasma_screening_lammps.mp4", writer="ffmpeg", fps=20,
             dpi=120, savefig_kwargs={"facecolor": "#030b14"})
    print("✓ Animation saved → plasma_screening_lammps.mp4")
except Exception as e:
    print(f"  ffmpeg not available ({e}). Displaying inline instead.")

HTML(ani.to_jshtml())

---
## Cell 8 — Interactive Parameter Exploration

Re-run either simulation at a different temperature or density and see how the Debye length, coupling parameter, and g(r) change — mirroring the HTML simulator's slider controls.

Edit the parameters below and re-execute **Cell 1 → Cell 2 or 3 → Cell 4 → Cell 5 → Cell 6**.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  INTERACTIVE PARAMETER EXPLORER
#  Equivalent to the HTML simulator sliders.
#  Change values below and re-run Cell 1 → 2/3 → 4 → 5 → 6.
# ══════════════════════════════════════════════════════════════

# HTML range: 0.01 eV – 2.0 eV  (default 0.10 eV)
T_eV   = 0.10     # [eV]   temperature

# HTML range: 10^18 – 10^23 m⁻³  (default 10^21 m⁻³)
n_m3   = 1e21     # [m⁻³]  number density

# HTML range: 10 – 120  (default 50)
N_p    = 50       # proton count
N_e    = 50       # electron count

# ── Quick preview of parameters without running simulation ────
T_K_ex    = T_eV * 11604.52
lD_ex     = np.sqrt(eps0 * k_B * T_K_ex / (n_m3 * e**2))
A_ex      = (e**2 / (4*np.pi*eps0)) / (k_B * T_K_ex * lD_ex)
tau_ex    = np.sqrt(m_p * lD_ex**2 / (k_B * T_K_ex))
rWS_ex    = np.sqrt(BOX**2 / (np.pi * N_p))
Gamma_ex  = A_ex / rWS_ex

print("=" * 58)
print("  PARAMETER PREVIEW  (run Cell 1 → 2/3 → 4 → 5 → 6)")
print("=" * 58)
print(f"  Temperature      : {T_eV} eV  =  {T_K_ex:.1f} K")
print(f"  Number Density   : {n_m3:.1e} m⁻³")
print(f"  Debye Length λ_D : {lD_ex:.4e} m  =  {lD_ex*1e10:.4f} Å")
print(f"  Yukawa A (norm)  : {A_ex:.6f}")
print(f"  Coupling Γ       : {Gamma_ex:.5f}  "
      f"({'weakly' if Gamma_ex < 1 else 'strongly'} coupled)")
print(f"  Time unit τ      : {tau_ex:.4e} s  =  {tau_ex*1e12:.4f} ps")
print(f"  Yukawa dt        : {DT_YUK} τ  =  {DT_YUK*tau_ex*1e15:.4f} fs")
print(f"  Coulomb dt       : {DT_COL} τ  =  {DT_COL*tau_ex*1e15:.4f} fs")
print()
print("  → To run: update T_eV / n_m3 in Cell 1 and re-execute")
print("            Cell 1 → 2 (Yukawa) or 3 (Coulomb) → 4 → 5 → 6")

---
## Notes on HTML ↔ LAMMPS Correspondence

| HTML feature | LAMMPS equivalent |
|---|---|
| `YukSim` class, `_forces()` | `pair_style yukawa κ cutoff` |
| `ColSim._wca()` | `pair_style hybrid/overlay lj/cut … coul/cut` |
| Velocity Verlet integrator | `fix nve` |
| `_thermo()` every 20 steps | `fix temp/rescale 20 …` |
| Periodic boundaries (wrap) | `boundary p p p` |
| `dimension 2` box | `dimension 2` |
| `RDF.acc()` / `RDF.gr()` | `compute_rdf_pp / compute_rdf_ep` (Cell 5) |
| `renderYuk()` / `renderCol()` | scatter plots in Cell 6 |
| `renderRDF()` | `plot_rdf()` in Cell 6 |
| Video download (200 frames) | `animation.FuncAnimation` + ffmpeg (Cell 7) |
| Slider: Temperature | `T_eV` in Cell 1 / Cell 8 |
| Slider: Number Density | `n_m3` in Cell 1 / Cell 8 |
| Slider: Protons N | `N_p` in Cell 1 / Cell 8 |
| Slider: Electrons N | `N_e` in Cell 1 / Cell 8 |
| Mass ratio 1:100 | `mass 2 0.01` |
| σ_pp=0.30, σ_ep=0.05, σ_ee=0.04 | `pair_coeff … SIG_PP/EP/EE` |
| WCA ε=5 | `eps_wca=5.0` in `pair_coeff lj/cut` |
| Yukawa cutoff = 3.0 λ_D | `CUT_YUK = 3.0` |
| Coulomb cutoff = 4.0 λ_D | `CUT_COL = 4.0` |
| Yukawa dt = 0.001 τ | `DT_YUK = 0.001` |
| Coulomb dt = 0.0001 τ | `DT_COL = 0.0001` |

### Key physics notes

- The **Yukawa / Debye-Hückel** model treats electrons as a continuous screening background. The proton-proton RDF shows g(r) < 1 at small r (Yukawa repulsion creating an exclusion zone) and g(r) → 1 at large r (random gas).
- The **Explicit Coulomb** model shows electrons forming a cloud around each proton — the Debye cloud. The electron-proton RDF shows g(r) > 1 at small r (Debye peak), which is the direct MD signature of plasma screening.
- The WCA cores prevent classical electron-proton collapse (which would happen without a quantum pressure term). The e-p WCA σ = 0.05 λ_D ≈ 3.7 Å for typical white-dwarf conditions.
- The coupling parameter **Γ = A_norm / r_WS** characterizes whether the plasma is weakly (Γ ≪ 1) or strongly (Γ ≫ 1) coupled. White-dwarf interiors span both regimes.